# Project 1: How old are New York City's dogs? 🐕

Every dog owner in NYC is required by law to license their dog. The city publishes those licenses as open data, and each one includes the year the dog was born. In this notebook, I'll use that to answer a simple question:

**What's the typical birth year of a licensed dog in New York City?**

I'll calculate the mean, median, and mode of the dogs' birth years twice: once with pandas, and once "the hard way" using only plain Python. Then I'll draw a chart of the results, again using only plain Python.

## The data

**Source:** [NYC Dog Licensing Dataset](https://data.cityofnewyork.us/Health/NYC-Dog-Licensing-Dataset/nu7n-tubp), from the NYC Department of Health and Mental Hygiene, via NYC Open Data.

The full dataset has hundreds of thousands of licenses, so I downloaded the first 50,000 rows using this link:

https://data.cityofnewyork.us/resource/nu7n-tubp.csv?$limit=50000

and saved the file next to this notebook as `nyc_dog_licenses.csv`.

Each row is one license, with these columns:

| Column | Meaning |
| --- | --- |
| `animalname` | The dog's name |
| `animalgender` | `M` or `F` |
| `animalbirth` | The year the dog was born |
| `breedname` | The dog's breed |
| `zipcode` | The owner's ZIP code |
| `licenseissueddate` / `licenseexpireddate` | When the license starts and ends |
| `extract_year` | The year the city pulled this record from its licensing system |

The only numeric column I care about is **`animalbirth`**, the dog's birth year.

## Part 1: Using pandas

First, let's load the data and take a look.

In [1]:
import pandas as pd

DATA_FILE = "nyc_dog_licenses.csv"

dogs = pd.read_csv(DATA_FILE)
dogs.head()

,animalname,animalgender,animalbirth,breedname,zipcode,licenseissueddate,licenseexpireddate,extract_year
0,PAIGE,F,2014,American Pit Bull Mix / Pit Bull Mix,10035,2014-09-12T00:00:00.000,2017-09-12T00:00:00.000,2016
1,YOGI,M,2010,Boxer,10465,2014-09-12T00:00:00.000,2017-10-02T00:00:00.000,2016
2,ALI,M,2014,Basenji,10013,2014-09-12T00:00:00.000,2019-09-12T00:00:00.000,2016
3,QUEEN,F,2013,Akita Crossbreed,10013,2014-09-12T00:00:00.000,2017-09-12T00:00:00.000,2016
4,LOLA,F,2009,Maltese,10028,2014-09-12T00:00:00.000,2017-10-09T00:00:00.000,2016


In [2]:
print(f"Number of licenses: {len(dogs):,}")

Number of licenses: 50,000


### A first look at the birth years

Before calculating anything, let's check the range of birth years to make sure they look reasonable.

In [3]:
dogs["animalbirth"].describe()

count    50000.00000
mean      2013.09336
std         25.55911
min         18.00000
25%       2010.00000
50%       2014.00000
75%       2018.00000
max       2024.00000
Name: animalbirth, dtype: float64

Hmm 🤔 The smallest birth year is way too small to be real. Let's look at the lowest values:

In [4]:
dogs["animalbirth"].sort_values().head(15)

3163       18
21686      21
9797      200
47889     202
49764     202
47249     202
26867     221
5565      317
39864     519
5982      720
21493    1912
40086    1973
6666     1991
6421     1991
6478     1991
Name: animalbirth, dtype: int64

Some of these are clearly typos (a dog wasn't born in the year 18 or 200!), and a dog born in the early 1900s would be over 100 years old.

### Cleaning the data

I'll keep only the licenses where the birth year makes sense. My assumptions:

- A dog can't be born **after** the year its record was pulled (`extract_year`).
- A dog is very unlikely to be **more than 30 years old** (the oldest dogs on record were around that age).

In [5]:
MAX_DOG_AGE = 30

age_at_extract = dogs["extract_year"] - dogs["animalbirth"]
is_valid_age = (age_at_extract >= 0) & (age_at_extract <= MAX_DOG_AGE)

valid_dogs = dogs[is_valid_age]
birth_years = valid_dogs["animalbirth"]

print(f"Licenses before cleaning: {len(dogs):,}")
print(f"Licenses after cleaning:  {len(valid_dogs):,}")
print(f"Removed: {len(dogs) - len(valid_dogs):,}")

Licenses before cleaning: 50,000
Licenses after cleaning:  49,987
Removed: 13


Only a handful of rows were removed, so the cleaning won't change the overall picture much, but it does stop those typos from dragging the average down.

### Mean, median, and mode

In [6]:
pandas_mean = birth_years.mean()
pandas_median = birth_years.median()
pandas_modes = list(birth_years.mode())

print(f"Mean birth year:   {pandas_mean:.1f}")
print(f"Median birth year: {pandas_median}")
print(f"Mode birth year:   {pandas_modes}")

Mean birth year:   2013.4
Median birth year: 2014.0
Mode birth year:   [2014]


- The **mean** is the average birth year.
- The **median** is the middle value: half the dogs were born before it, and half after.
- The **mode** is the single most common birth year. (pandas returns a list, because there could be a tie.)

## Part 2: The hard way

Now let's do the same thing using only the Python standard library: no pandas, no `statistics` module, and no built-in "mean"/"median"/"mode" functions. This section doesn't use anything from Part 1, so it would still work in a separate notebook.

### Reading the file

I'll use Python's built-in [`csv` module](https://docs.python.org/3/library/csv.html). Splitting each line on commas myself wouldn't be safe: some dog names in this file actually contain commas!

In [7]:
import csv

DATA_FILE = "nyc_dog_licenses.csv"
MAX_DOG_AGE = 30

hard_way_years = []

with open(DATA_FILE, newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        birth_year = int(row["animalbirth"])
        extract_year = int(row["extract_year"])
        age = extract_year - birth_year

        # same cleaning rule as in Part 1
        if 0 <= age <= MAX_DOG_AGE:
            hard_way_years.append(birth_year)

print(f"Valid birth years read: {len(hard_way_years):,}")

Valid birth years read: 49,987


That should match the number of rows pandas kept after cleaning. ✅

### Mean

The mean is the total of all the values, divided by how many values there are.

In [8]:
total = 0
count = 0
for year in hard_way_years:
    total = total + year
    count = count + 1

hard_way_mean = total / count
print(f"Mean birth year: {hard_way_mean:.1f}")

Mean birth year: 2013.4


### Median

To find the median, sort the values and take the one in the middle. If there's an even number of values, there are two "middle" values, so we take the average of those two.

In [9]:
sorted_years = sorted(hard_way_years)
num_years = len(sorted_years)
middle = num_years // 2

if num_years % 2 == 1:
    hard_way_median = sorted_years[middle]
else:
    hard_way_median = (sorted_years[middle - 1] + sorted_years[middle]) / 2

print(f"Median birth year: {hard_way_median}")

Median birth year: 2014


### Mode

To find the most common value, I'll use a [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) to count how many dogs were born in each year. Then I'll find the highest count, and every year that has that count.

In [10]:
year_counts = {}
for year in hard_way_years:
    if year in year_counts:
        year_counts[year] = year_counts[year] + 1
    else:
        year_counts[year] = 1

highest_count = 0
for year in year_counts:
    if year_counts[year] > highest_count:
        highest_count = year_counts[year]

hard_way_modes = []
for year in year_counts:
    if year_counts[year] == highest_count:
        hard_way_modes.append(year)

print(f"Mode birth year: {sorted(hard_way_modes)} ({highest_count:,} dogs)")

Mode birth year: [2014] (5,786 dogs)


### Do the two methods agree?

In [11]:
print(f"Mean:   pandas = {pandas_mean:.1f}, hard way = {hard_way_mean:.1f}")
print(f"Median: pandas = {pandas_median}, hard way = {hard_way_median}")
print(f"Mode:   pandas = {pandas_modes}, hard way = {sorted(hard_way_modes)}")

Mean:   pandas = 2013.4, hard way = 2013.4
Median: pandas = 2014.0, hard way = 2014
Mode:   pandas = [2014], hard way = [2014]


They match! 🎉 pandas saved us a lot of typing, but under the hood it's doing the same kind of work.

## Part 3: Visualization

Numbers are nice, but a picture shows the whole distribution at once. Below is a bar chart of how many licensed dogs were born in each year, drawn with nothing but `print()`.

To avoid printing thousands of blocks, each bar is scaled so that the most common year gets the longest bar, and everything else is drawn in proportion.

In [12]:
dogs_per_year = birth_years.value_counts().sort_index()

MAX_BAR_LENGTH = 40
largest_count = dogs_per_year.max()
dogs_per_block = largest_count / MAX_BAR_LENGTH

print("🐕 Licensed NYC dogs, by birth year")
print(f"(each █ is about {dogs_per_block:,.0f} dogs)")
print()

for year, num_dogs in dogs_per_year.items():
    bar_length = round(num_dogs / largest_count * MAX_BAR_LENGTH)
    bar = "█" * bar_length
    print(f"{year}: {bar} {num_dogs:,}")

🐕 Licensed NYC dogs, by birth year
(each █ is about 145 dogs)

1991:  8
1992:  7
1993:  15
1994:  11
1995:  24
1996:  51
1997: █ 92
1998: █ 108
1999:  1
2000: ███ 414
2001: ████ 563
2002: █████ 680
2003: ██████ 831
2004: ██████ 869
2005: ████████ 1,120
2006: █████████ 1,230
2007: ███████████ 1,561
2008: ████████████ 1,688
2009: ███████████████ 2,108
2010: █████████████████ 2,447
2011: ███████████████████ 2,752
2012: ██████████████████████ 3,236
2013: ███████████████████████████ 3,912
2014: ████████████████████████████████████████ 5,786
2015: ██████████████████████████ 3,816
2016: ██████████ 1,412
2017: ███████████ 1,578
2018: ██████████████ 1,967
2019: █████████████████ 2,490
2020: █████████████████████████ 3,545
2021: ██████████████████ 2,643
2022: ████████████ 1,712
2023: █████████ 1,304
2024:  6


## What did we learn?

- The typical licensed dog in this data was born in the early-to-mid 2010s. The mean, median, and mode all land close together.
- The chart isn't a smooth hill. That's because of how this sample was pulled: most of these rows come from two snapshots of the licensing system, one taken in 2016 and one in 2026. Each snapshot mostly contains dogs that were young at the time, so we see two bumps instead of one.
- There are very few dogs with birth years in the 1990s, which makes sense: those dogs would be getting old (or no longer around) by the time their license was recorded.

### Caveats

- Each row is a **license**, not a unique dog. A dog that renewed its license may be counted more than once.
- I only used the first 50,000 rows of the dataset, not the whole thing.
- Only licensed dogs are included. Plenty of NYC dogs are probably never licensed, so this isn't a full picture of every dog in the city.
- My cleaning rule (age 0 to 30) is my own judgment call.